# 共鳴の Max-Cut を、タブレットから IBM の量子機へ（道 B）

六つのコイン＝六つの量子ビット、QAOA 1 層。`/mente/game/root-bloom` と同じ問題（重み 1 − 相関、値段は使わない）。投資の助言ではない。

## 安全の決まり（実行の前に読む）
1. **鍵はノートに書かない。** 左の 🔑「シークレット」に `IBM_QUANTUM_TOKEN` と `IBM_QUANTUM_INSTANCE` を保存し、「ノートブックからのアクセス」はこのノートだけに許可する。鍵を画面に表示しない、変数にも残さない。
2. **入れるものを確かめる。** パッケージ（wheel）の SHA-256 が、下の `EXPECTED` と同じときだけ入れる。ライブラリの版も固定する。
3. **実機は二重の確認。** `RUN_ON_QPU = True` かつ `CONFIRM = "RUN ON QPU"` のときだけ送る。送る前に回路の大きさが表示される。ショット数は 10,000 が上限。
4. **残さない。** 結果の JSON には鍵もアカウントも入らない。終わったら「ランタイム → ランタイムを接続解除して削除」。このノートを他人と共有しない。

## 1. wheel を選んで、指紋（SHA-256）を確かめる
リポジトリの `quantum/dist/cure_oracle-0.2.0-py3-none-any.whl` を、タブレットに保存してから選ぶ。

In [ ]:
import hashlib, pathlib
from google.colab import files
EXPECTED = "2e72d8272d06a125f1fbb67c341706c74d5ae7f363a50988d11c1ec9591e690d"  # quantum/dist/SHA256SUMS と同じ値
up = files.upload()
name = next(iter(up)); digest = hashlib.sha256(up[name]).hexdigest()
assert digest == EXPECTED, f"指紋が違う: {digest} — 入れない"
pathlib.Path(name).write_bytes(up[name]); print("OK", name)

## 2. 版を固定して入れる（このパッケージは依存なしで入れ、Qiskit は確かめた版で）

In [ ]:
%pip install -q "qiskit==2.6.2" "qiskit-ibm-runtime==0.52.1" "qiskit-aer==0.18.2"
%pip install -q --no-deps ./cure_oracle-0.2.0-py3-none-any.whl

## 3. 実行（無料・オフライン）：手元の理想と、偽のチップのリハーサル

In [ ]:
from cure_oracle import resonance
local = resonance.run("local")
fake = resonance.run("fake")

## 4. 実行（実機・無料枠の時間を使う）：二重の確認つき
実行する日だけ、`RUN_ON_QPU = True` と `CONFIRM = "RUN ON QPU"` に書き換える。

In [ ]:
RUN_ON_QPU = False   # ← 実行する日だけ True
CONFIRM = ""         # ← "RUN ON QPU" と打つ
if RUN_ON_QPU:
    from google.colab import userdata
    qpu = resonance.run("ibm", shots=2000, confirm=CONFIRM,
                        token=userdata.get("IBM_QUANTUM_TOKEN"), instance=userdata.get("IBM_QUANTUM_INSTANCE"))
    print("QPU 秒:", qpu["qpu_seconds"])
else:
    print("実機には送っていない")

## 5. 結果をタブレットに保存して、ChartUI に載せる
実機の結果は `resonance-ibm.json` という名前にして `public/quantum/` に置くと、`/mente/game/root-bloom` の「記録の再生」に出る。

In [ ]:
import glob, shutil
for p in glob.glob("resonance-ibm-*.json"):
    shutil.copy(p, "resonance-ibm.json")
for p in glob.glob("resonance-*.json"):
    files.download(p)